In [1]:
import pandas as pd
import numpy as np


from openai import OpenAI
df= pd.read_csv('top_rated_wines.csv' )
df = df[df["variety"].notna()]
data= df.sample(frac=0.5).to_dict(orient='records')
print(data)


[{'name': 'Chateau de Beaucastel Hommage Jacques Perrin Chateauneuf-du-Pape 2009', 'region': 'Chateauneuf-du-Pape, Rhone, France', 'variety': 'Red Wine', 'rating': 97.0, 'notes': 'Deep black-ruby color. Profound aromas of black cherry, cassis, spice, leather and game, with an almost medicinal aspect. Very sweet entry, then firm and closed, almost too hard on the palate today. Extremely concentrated on the finish. This is a wine to be kept for your retirement.'}, {'name': 'Chateau Suduiraut Sauternes 2009', 'region': 'Sauternes, Bordeaux, France', 'variety': 'Collectible', 'rating': 96.0, 'notes': 'This is a precise yet opulent Suduiraut with the purest of fruit, perfectly-integrated sweetness and a long finish revealing a beautiful acidity. The attack is rich and crisp at the same time. The palate reveals candied citrus flavours, followed by wood and liquorice, leading through perfectly into a smooth, fruity finish. In this very sunny vintage, we can sense very great potential, with 20

In [7]:
from qdrant_client import models, QdrantClient
from sentence_transformers import SentenceTransformer

In [8]:

encoder = SentenceTransformer("all-MiniLM-L6-v2") # Model to create embeddings

In [4]:
quadrant= QdrantClient(":memory:") # Create in-memory Qdrant instance

In [ ]:
# Create collection to store wines
if quadrant.collection_exists("top_wines"):
    quadrant.delete_collection("top_wines")
quadrant.create_collection(
    collection_name="top_wines",
    #size = encoder.get_sentence_embedding_dimension() if encoder.get_sentence_embedding_dimension() is not None else 0,  # Vector size is defined by used model
    vectors_config=models.VectorParams(
        size = encoder.get_sentence_embedding_dimension() if encoder.get_sentence_embedding_dimension() is not None else 0,  # Vector size is defined by used model
        distance=models.Distance.COSINE
    )
)

AssertionError: Unknown arguments: ['size']

In [20]:
# vectorize!
quadrant.upload_points(
    collection_name="top_wines",
    points=[
        models.PointStruct(
            id=idx,
            vector=encoder.encode(doc["notes"]).tolist(),
            payload=doc,
        ) for idx, doc in enumerate(data) # data is the variable holding all the wines
    ]
)

In [21]:
user_prompt = "Which wines are similar to a fruity and soft Pinot Noir?"

In [22]:
hits = quadrant.search(
        collection_name="top_wines",
        query_vector=encoder.encode(user_prompt).tolist(), # Create embedding for user prompt
        limit=3, # Return 3 most similar wines
)
for hit in hits:
    payload = (hit.payload,"score: ", hit.score)
    print(payload)

({'name': 'Hartford Court Far Coast Pinot Noir 2016', 'region': 'Sonoma Coast, Sonoma County, California', 'variety': 'Red Wine', 'rating': 96.0, 'notes': 'Far Coast Vineyard consistently produces one of our boldest Pinot Noir offerings. The 2016 follows suit with intense cherry cola, blackberry and cocoa aromas. The subtle earth and underbrush characters carry into the flavors along with Asian spice and a broad, lush texture followed by black cherry and blackberry fruits. The persistent finish is distinctly Far Coast.'}, 'score: ', 0.7172731979110145)
({'name': "Hartford Court Jennifer's Pinot Noir 2016", 'region': 'Russian River, Sonoma County, California', 'variety': 'Red Wine', 'rating': 96.0, 'notes': "Jennifer's Pinot noir is both elegant and powerful.  It features aromas of boysenberries, violets and lavender followed by flavors of black raspberries, black tea, and anise.  The 2016 Jennifer's Pinot is very dense, but with fine tannins and a silky, persistent mouthfeel.  This win

C:\Users\arr12\AppData\Local\Temp\ipykernel_45396\2200323205.py:1: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  hits = quadrant.search(


In [24]:
search_result = [hit.payload for hit in hits]

In [26]:
from openai import OpenAI
client = OpenAI(
    base_url="http://127.0.0.1:8080/v1", # "http://<Your api-server IP>:port"
    api_key = "sk-no-key-required"
)
completion = client.chat.completions.create(
    model="LLaMA_CPP",
    messages=[
        {"role": "system", "content": "You are chatbot, a wine specialist. Your top priority is to help guide users into selecting amazing wine and guide them with their requests."},
        {"role": "user", "content": "Suggest me an amazing Malbec wine from Argentina"},
        {"role": "assistant", "content": str(search_result)}
    ]
)
print(completion.choices[0].message)

APIConnectionError: Connection error.